In [38]:
import xgboost as xgb
import pandas as pd
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import cross_val_score, KFold, train_test_split, TimeSeriesSplit
from mlxtend.evaluate import GroupTimeSeriesSplit
from sklearn.metrics import r2_score

In [95]:
df = pd.read_csv('../data_preprocessing/dataset_1_pixels_grid_indices.csv')
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(['row', 'col', 'date'])

df = df.sort_values('date').reset_index(drop=True)

In [96]:
print(df.head())

   row  col       date  pm25_today  u_wind_10m  v_wind_10m  dew_temp_2m  \
0    0    0 2018-07-04    8.014978    1.901375    0.232695    290.75000   
1   35    8 2018-07-04    8.911477    1.885099    0.797799    292.50668   
2   35    7 2018-07-04    8.015140    1.673469    0.863839    292.54315   
3    5   33 2018-07-04    8.458900    1.958992    1.423776    294.79535   
4   35    6 2018-07-04    7.435858    1.360278    0.935901    293.76830   

     temp_2m  surf_pressure  precip_sum  frp  elevation  pm25_change  
0  294.50488       85355.38    0.004142  0.0     1360.0     0.302968  
1  294.42984       89309.88    0.004603  0.0     1151.0     1.609836  
2  293.79590       89010.13    0.007146  0.0     1250.0     2.941129  
3  298.75610       92667.51    0.001118  0.0      794.0     0.702901  
4  294.79434       92103.22    0.009098  0.0      824.0     2.549173  


In [104]:
# df_filtered = df[df['date'].dt.month.isin([1, 2, 3, 4, 5])]
df_filtered = df

dates = df_filtered["date"].unique()

train_days = 800
gap_days = 14
test_days = 28 * 2

train_start = 0

num_days = len(dates)
i = 1

errors = []
while True:
    train_end = train_start + train_days
    test_start = train_end + gap_days
    test_end = test_start + test_days

    if test_end > num_days:
        break

    train_dates = dates[train_start: train_end]
    test_dates = dates[test_start: test_end]

    print(f'Fold {i}')
    print(train_dates.min(), train_dates.max())
    print(test_dates.min(), test_dates.max())

    train_data =  df[df["date"].isin(train_dates)]
    test_data = df[df["date"].isin(test_dates)]

    X_train = train_data.drop(['date', 'pm25_change'], axis=1)
    y_train = train_data['pm25_change']

    X_test = test_data.drop(['date', 'pm25_change'], axis=1)
    y_test = test_data['pm25_change']

    model = xgb.XGBRegressor(eta=0.1)
    model.fit(X_train, y_train)

    y_preds = model.predict(X_test)

    # y_preds = np.zeros_like(y_test)

    mse = mean_squared_error(y_test, y_preds)
    rmse = root_mean_squared_error(y_test, y_preds)
    r2 = r2_score(y_test, y_preds)

    errors.append(r2)

    print(f"MSE: {mse}")
    print(f"RMSE: {rmse}")
    print(f"R^2: {r2}")
    train_start += test_days
    i += 1

print(np.mean(errors))


Fold 1
2018-07-04 00:00:00 2020-09-26 00:00:00
2020-10-11 00:00:00 2020-12-05 00:00:00
MSE: 26.392814901541477
RMSE: 5.13739378494013
R^2: 0.2760769161968306
Fold 2
2018-08-29 00:00:00 2020-11-21 00:00:00
2020-12-06 00:00:00 2021-01-30 00:00:00
MSE: 49.19813610850506
RMSE: 7.014138301210282
R^2: 0.09047544793590367
Fold 3
2018-10-24 00:00:00 2021-01-16 00:00:00
2021-01-31 00:00:00 2021-04-08 00:00:00
MSE: 172.65678158885808
RMSE: 13.139892754085098
R^2: 0.10417971510406443
Fold 4
2018-12-19 00:00:00 2021-03-25 00:00:00
2021-04-09 00:00:00 2021-06-03 00:00:00
MSE: 31.176938126879453
RMSE: 5.583631267094869
R^2: 0.2021045637979738
Fold 5
2019-02-15 00:00:00 2021-05-20 00:00:00
2021-06-04 00:00:00 2021-07-29 00:00:00
MSE: 1.3029436764878368
RMSE: 1.1414655826996436
R^2: 0.050357885912792444
Fold 6
2019-04-12 00:00:00 2021-07-15 00:00:00
2021-07-30 00:00:00 2021-09-23 00:00:00
MSE: 1.2340563219920695
RMSE: 1.110880876598418
R^2: -0.11353080350065681
Fold 7
2019-06-07 00:00:00 2021-09-09 00